# Implementing GPT model from scratch to generate text

In [49]:
! pip3 install torch
! pip3 install tiktoken

In [50]:
import torch
import torch.nn as nn
import tiktoken

torch.set_printoptions(sci_mode=False, precision=4)

# Feed forward with GeLU activation

In [51]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

In [52]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # Expansion
        GELU(), # Activation
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # Contraction
    )

  def forward(self, x):
    return self.layers(x)

## MultiHead Attention

In [53]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    # calculate individual head dimension according to d_out and no. of heads present
    self.head_dim = d_out // num_heads

    # random key,query,value initialization with d_in and d_out)
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_in, d_out) # Linear layer to combine head outputs
    self.dropout = nn.Dropout(dropout)

    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape # initialize (Batch, token_size, input_dimension)

    # keys, values queries (random of d_in,d_out(dimensions) multiplied with inputs)
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # convert of each head i.e d_out --> num_heads and head_dimension
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Group matrices by num_heads for parallel computation.

    #(b,num_tokens,num_heads,head_dim) --> (b, num_heads, num_tokens, head_dim)
    # (1,3,2,3) --> (1,2,3,3) (The positions 1 and 2 will be transposed)
    keys = keys.transpose(1,2)
    queries = queries.transpose(1,2)
    values = values.transpose(1,2)

    # now for each query we will do matmul with keys.
    # and for that we need to transpose the postion 2 and 3 of keys.
    # (b,num_heads,num_tokens,head_dim) * (b, num_heads, head_dim, num_tokens)
    #                                    |
    #                    (b,num_heads,num_tokens,num_tokens)
    attn_scores = queries @ keys.transpose(2,3)

    # masking
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

    # softmax with Sqrt of head_dim and dropout
    attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # calulate context vector with d_out as dimension preserved

    # (b,num_heads,num_tokens,num_tokens) * (b,num_heads,num_tokens,head_dim)
    #                                     |
    #                     (b,num_heads,num_tokens,head_dim)
    #                                     | (1,2) transpose
    #                     (b,num_tokens,num_heads,head_dim)
    context_vector = (attn_weights @ values).transpose(1,2)
    # now we can merge num_heads and head_dim easily to d_out.
    # we merge the num_heads and head_dim into single row giving d_out dimension.
    # (b,num_tokens,num_heads,head_dim) --> (b,num_tokens,d_out)
    # contiguous ensures that after reshaping the values stay in same block of memory.
    context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
    context_vector = self.out_proj(context_vector)

    return context_vector

# **GPT architecture**

In [54]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Use a placeholder for transformer block
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    # Use a placeholder for layer norm
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg["emb_dim"],
        context_length = cfg["context_length"],
        num_heads = cfg["n_heads"],
        dropout = cfg["drop_rate"],
        qkv_bias = cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # shortcut connection for attention block
    shortcut = x
    x = self.norm1(x) # normalization
    x = self.att(x) # attention
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    # shortcut connection for feed forward block
    shortcut = x
    x = self.norm2(x) # normalization
    x = self.ff(x) # feed forward
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    return x

class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    # if unbiased is 'True', it applied Bessels correction which is divide by n-1 for variance not by n.
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps) # eps --> Epsilon is used to prevent division by 0 during normalization.
    return self.scale * norm_x + self.shift # scale and shifts are trainable parameters used to tweak norms.

# Using GPT to generate output

In [55]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
  for _ in range(max_new_tokens):
    # crop current context if it exceeds the supported context size
    # e.g. if LLM supports only 5 tokens, and the context_size is 10
    # then only the last 5 tokens are used as context to predict next word
    idx_cond = idx[:, -context_size:]

    # Get the predictions
    with torch.no_grad():
      logits = model(idx_cond) # batch, n_tokens, vocab_size

    # Focus only on the last row from each batches
    # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
    logits = logits[:, -1, :]

    # Apply softmax to get the probabilities
    probas = torch.softmax(logits, dim=-1) # batch,vocab_size

    # Get the index with highest probability
    idx_next = torch.argmax(probas, dim=-1, keepdim=True) # (batch, 1)

    # Append sampled index to the running sequence.
    idx = torch.cat((idx, idx_next), dim=1) # (batch, n_tokens+1)

  return idx

In [56]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 256, # Shortened context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,  # no. of attention heads
    "n_layers": 12, # no. of transformer layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval(); # Disable dropout during inference

In [57]:
def text_to_token_ids(text, tokenizer):
  encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
  encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat = token_ids.squeeze(0) # remove batch dimension
  decoded = tokenizer.decode(flat.tolist())
  return decoded

# Using story dataset

In [58]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 256, # Shortened context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,  # no. of attention heads
    "n_layers": 12, # no. of transformer layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
tokenizer = tiktoken.get_encoding("gpt2")

In [59]:
import os
import urllib.request

file_path = "story_dataset.txt"
url = "https://raw.githubusercontent.com/utsab818/ThisIsGPT/refs/heads/main/story_dataset.txt"

if not os.path.exists(file_path):
  with urllib.request.urlopen(url) as response:
    text_data = response.read().decode('utf-8')
  with open(file_path, 'w', encoding="utf-8") as file:
    file.write(text_data)
else:
  with open(file_path, 'r', encoding="utf-8") as file:
    text_data = file.read()

In [60]:
print(text_data[:150])

The Silent Valley and the Keepers of Its Voice

Part One: The Weaver's Awakening

In the ancient cradle of the world, where time itself seemed to bend


In [61]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print(f"Total characters: {total_characters}")
print(f"Total tokens: {total_tokens}")

Total characters: 24361
Total tokens: 5751


### Using dataset and dataloader

In [62]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # use sliding window with max_length as sequence length
    # since we will be sliding by 4 words, the stride here will be 4
    # which means we skip 4 words to create next sequence.
    for i in range(0, len(token_ids)-max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [63]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

  # Initialize the tokenizer
  tokenizer = tiktoken.get_encoding("gpt2")

  # Initialize the dataset
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  # Create the dataloader which makes batch processing easier
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          drop_last=drop_last, num_workers=num_workers)

  return dataloader

In [64]:
# Train/Validation Ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))

# Split the data
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [65]:
# Sanity check
if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
  print("Not enough tokens for the training loader. "
        "Try to lower the `GPT_CONFIG_124M['context_length']` or "
        "increase the `train_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
  print("Not enough tokens for the validation loader. "
        "Try to lower the `GPT_CONFIG_124M['context_length']` or "
        "decrease the `train_ratio`")

In [66]:
print("Train loader:")
for x,y in train_loader:
  print(x.shape, y.shape)

print("\nValidation loader:")
for x,y in val_loader:
  print(x.shape, y.shape)

print(len(train_loader))
print(len(val_loader))

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])
10
1


In [67]:
train_tokens = 0
for input_batch, target_batch in train_loader:
  train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
  val_tokens += input_batch.numel()

print("Training tokens: ", train_tokens)
print("Validation tokens: ", val_tokens)
print("All tokens: ", train_tokens + val_tokens)

Training tokens:  5120
Validation tokens:  512
All tokens:  5632


In [68]:
def calc_loss_batch(input_batch, target_batch, model, device):
  input_batch = input_batch.to(device)
  target_batch = target_batch.to(device)
  logits = model(input_batch)
  loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
  return loss

# the above calc_loss_batch is for single batch, but we have many batches and we will aggregate it all and pass the above function.
def calc_loss_loader(data_loader, model, device, num_batches=None):
  total_loss = 0
  if len(data_loader) == 0:
    return float("nan")
  elif num_batches is None:
    num_batches = len(data_loader)
  else: # if num_batches exceeds the no. of batches in dataloader
    num_batches = min(len(data_loader), num_batches)

  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      loss = calc_loss_batch(input_batch, target_batch, model, device)
      total_loss += loss
    else:
      break

  return total_loss / num_batches

In [69]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("CUDA is available")
elif torch.backends.mps.is_available():
  print("MPS is available")
  device = torch.device("mps")
else:
  device = torch.device("cpu")
  print("CPU is available")

model.to(device)

torch.manual_seed(123)

with torch.no_grad():
  train_loss = calc_loss_loader(train_loader, model, device)
  val_loss = calc_loss_loader(val_loader, model,device)

print("Training loss: ", train_loss.item())
print("Validation loss: ", val_loss.item())

MPS is available
Training loss:  10.991082191467285
Validation loss:  10.96342658996582


<h1> Training </h1>

In [70]:
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    # initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # main training loop
    for epoch in range(num_epochs):
        model.train() # set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # calculate loss gradients
            optimizer.step() # update model weights using loss gradients
            tokens_seen += input_batch.numel() # returns the total no. of tokens
            global_step += 1

            # optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(
            model, tokenizer, device, start_context
        )
    return train_losses, val_losses, track_tokens_seen

In [71]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [72]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [73]:
# Train model with 10 epochs using AdamW optimizer
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
# specify learning rate(lr) and weight decay in optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1) 

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="In the ancient cradle", tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 9.803, Val loss 9.739
Ep 1 (Step 000005): Train loss 8.074, Val loss 8.429
In the ancient cradle,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Ep 2 (Step 000010): Train loss 6.857, Val loss 7.521
Ep 2 (Step 000015): Train loss 6.139, Val loss 7.307
In the ancient cradle, the, the. ", the, the, the, the, the, the, the, the, the, the, the, the, the, the, the. ", the, the, the, the, the, the
Ep 3 (Step 000020): Train loss 5.586, Val loss 7.274
Ep 3 (Step 000025): Train loss 5.765, Val loss 7.412
In the ancient cradle, but the valley.                                             
Ep 4 (Step 000030): Train loss 4.963, Val loss 7.283
Ep 4 (Step 000035): Train loss 4.721, Val loss 7.297
In the ancient cradle, the the, the.                                            
Ep 5 (Step 000040): Train loss 4.327, Val loss 7.158
Ep 5 (Step 000045): Train loss 3.676, Val loss 7.174
In the ancient cradle, and the air. The valley's voice like a leaf. The valley's voice, an

You can see the validation loss is stagnated to 7.7 or something. This is the pure case of overfitting as we can see memorization.
This is expected in small datasets.

<h1>Temperature Scaling</h1>

In [74]:
vocab = {
"closer": 0,
"every": 1,
"effort": 2,
"forward": 3,
"inches": 4,
"moves": 5,
"pizza": 6,
"toward": 7,
"you": 8,
}

inverse_vocab = {v: k for k, v in vocab.items()}

In [75]:
# output will be of 9 as we have 9 vocabs
next_token_logits = torch.tensor(
[4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

In [76]:
probas = torch.softmax(next_token_logits, dim=0)
print(probas)
next_token_id = torch.argmax(probas).item()
print(inverse_vocab[next_token_id])

tensor([0.0609, 0.0016, 0.0001, 0.5721, 0.0034, 0.0001, 0.0001, 0.3576, 0.0040])
forward


In [77]:
# To implement a probabilistic sampling process, we can now replace argmax with
# the multinomial function in PyTorch
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

forward


In [78]:
def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item()
    for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")
print_sampled_tokens(probas)

73 x closer
0 x every
0 x effort
582 x forward
2 x inches
0 x moves
0 x pizza
343 x toward


In [79]:
# Temperature scaling is just a fancy description for dividing the logits by a number greater than 0
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

<h1> Top-k Sampling </h1>

With temperature scaling, we provide every tokens opportunity to be the next token, but it's not feasible to let every tokens be the next token.
So, to restrict the no. of tokens to be the part of next token, we use top-k sampling.
In top-k sampling, we take top k values as a group which will take part in next token.
We can achieve this top-k values by coverting every other token values except top-k to -inf, which when applied softmax gives value of 0.

In [80]:
next_token_logits = torch.tensor(
[4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

In [81]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
print("Top logits:", top_logits)
print("Top positions:", top_pos)
# Here you can see that top 3 values are returned who will take part in next token.

Top logits: tensor([6.7500, 6.2800, 4.5100])
Top positions: tensor([3, 7, 0])


In [82]:
# Now replace all other values with -inf who does not belong to top logits.
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float('-inf')),
    other=next_token_logits
)
print(new_logits)

tensor([4.5100,   -inf,   -inf, 6.7500,   -inf,   -inf,   -inf, 6.2800,   -inf])


In [83]:
# Now apply softmax
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

tensor([0.0615, 0.0000, 0.0000, 0.5775, 0.0000, 0.0000, 0.0000, 0.3610, 0.0000])


Logits -> Top-k sampling -> Logits/Temperature -> Softmax -> Sample from multinomial

In [87]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):

    device = next(model.parameters()).device
    idx = idx.to(device)

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        # ---- Top-K Filtering ----
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1].unsqueeze(-1)

            logits = torch.where(
                logits < min_val,
                torch.full_like(logits, float('-inf')),
                logits
            )

        # ---- Sampling ----
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # ---- EOS Handling ----
        if eos_id is not None:
            if (idx_next == eos_id).any():
                break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [88]:
torch.manual_seed(123)
token_ids = generate(
model=model,
    idx=text_to_token_ids("In the ancient cradle", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 In the ancient cradle hear— ward off shadows, Miraows, painting the Weaver's lo
